[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C13_RL_Foundations_Course/05_bandits_exploration/05_bandits_exploration.ipynb)

# 05 · 多臂老虎机与探索（纯 numpy）

目标：从零实现 **ε-greedy**、**UCB1**、**Thompson sampling**，定义并测量 **regret**，正面对比三者，用干净判据验证「贪心/固定ε 线性 regret」vs「UCB/Thompson 次线性 regret」。

路线：伯努利老虎机 + regret → ε-greedy(固定vs衰减) → UCB1(置信半径推导) → Thompson(Beta后验) → 三策略 regret 对比 → ✏️ 练习 → 📖 答案 → 🧪 A/B 测试胶囊。

> 心智模型：**explore-exploit**——拉「看起来最好的」(利用) vs 拉「没试够、可能更好的」(探索)。好策略 regret 次线性(每步代价趋 0)，盲目策略线性(永交探索税)。

## 1 · 伯努利老虎机与 regret

$K$ 个臂，臂 $a$ 以未知概率 $\mu_a$ 给奖励 1(否则 0)。regret $= T\mu^* - \mathbb{E}[\sum r_t] = \sum_a \Delta_a N_a(T)$，其中 $\Delta_a = \mu^* - \mu_a$ 是次优间隙。先搭老虎机和 regret 度量。

In [ ]:
import numpy as np

class BernoulliBandit:
    def __init__(self, true_means):
        self.means = np.asarray(true_means, dtype=float)
        self.K = len(self.means)
        self.mu_star = self.means.max()
    def pull(self, a, rng):
        return 1.0 if rng.random() < self.means[a] else 0.0
    def gap(self, a):
        return self.mu_star - self.means[a]      # 拉臂 a 每次的期望 regret

true_means = np.array([0.2, 0.5, 0.7, 0.4, 0.55])   # 臂 2 最优(0.7)
bandit = BernoulliBandit(true_means)
print(f'{bandit.K} 个臂，真实中奖率 {true_means}，最优臂 {true_means.argmax()} (μ*={bandit.mu_star})')
print('各臂次优间隙 Δ_a:', np.round([bandit.gap(a) for a in range(bandit.K)], 2))
# 拉最优臂不增 regret，拉最差臂(臂0)每次增 0.5
assert bandit.gap(2) == 0.0, '最优臂间隙为 0'
assert abs(bandit.gap(0) - 0.5) < 1e-9, '最差臂间隙 = 0.7-0.2'
print('✅ 伯努利老虎机与 regret 度量就绪')

## 2 · 纯贪心的灾难：线性 regret

纯贪心(只拉当前估计最高的臂、从不探索)会怎样？初期运气不好可能**永久卡在次优臂**。

我们跑多个种子，看它的累积 regret——会发现它线性爆炸，且高度依赖运气。

In [ ]:
def run_strategy(bandit, strategy, T=5000, seed=0, eps=0.1):
    '''统一的老虎机循环。返回 (rewards, cumulative_regret, counts)。'''
    rng = np.random.default_rng(seed)
    K = bandit.K
    counts = np.zeros(K); values = np.zeros(K)
    alpha = np.ones(K); beta = np.ones(K)    # Thompson 的 Beta 后验
    cum_regret = 0.0; regrets = []; rewards = []
    for t in range(1, T + 1):
        if strategy == 'greedy':
            a = int(np.argmax(values))
        elif strategy == 'epsilon':
            a = int(rng.integers(K)) if rng.random() < eps else int(np.argmax(values))
        elif strategy == 'epsilon_decay':
            e = min(1.0, K / t)              # 衰减 ε ~ 1/t
            a = int(rng.integers(K)) if rng.random() < e else int(np.argmax(values))
        elif strategy == 'ucb':
            if (counts == 0).any():
                a = int(np.argmin(counts))   # 先把每个臂拉一次
            else:
                a = int(np.argmax(values + np.sqrt(2 * np.log(t) / counts)))
        elif strategy == 'thompson':
            a = int(np.argmax(rng.beta(alpha, beta)))
        r = bandit.pull(a, rng)
        counts[a] += 1
        values[a] += (r - values[a]) / counts[a]    # 增量平均
        if strategy == 'thompson':
            alpha[a] += r; beta[a] += (1 - r)
        cum_regret += bandit.gap(a)
        regrets.append(cum_regret); rewards.append(r)
    return np.array(rewards), np.array(regrets), counts

greedy_regrets = [run_strategy(bandit, 'greedy', T=5000, seed=sd)[1][-1] for sd in range(20)]
print(f'纯贪心最终累积 regret: 均值={np.mean(greedy_regrets):.0f}, '
      f'范围=[{min(greedy_regrets):.0f}, {max(greedy_regrets):.0f}]')
# 贪心 regret 大且方差大(取决于初期运气)
assert np.mean(greedy_regrets) > 500, '纯贪心 regret 应很大(线性、卡次优臂)'
print('✅ 纯贪心灾难：regret 线性爆炸、且高度依赖运气(初期失算就永久卡死)')

## 3 · ε-greedy：固定 vs 衰减

固定 ε 永远花 ε 比例随机探索 → 线性 regret(永交探索税)。衰减 ε(~1/t)探索随时间减少 → 接近对数。

对比两者的最终 regret。

In [ ]:
eps_fixed = [run_strategy(bandit, 'epsilon', T=5000, seed=sd, eps=0.1)[1][-1] for sd in range(20)]
eps_decay = [run_strategy(bandit, 'epsilon_decay', T=5000, seed=sd)[1][-1] for sd in range(20)]
print(f'固定 ε=0.1 最终 regret = {np.mean(eps_fixed):.1f}')
print(f'衰减 ε~1/t 最终 regret = {np.mean(eps_decay):.1f}')
# ε-greedy 远好于纯贪心(有探索，不会永久卡死)
assert np.mean(eps_fixed) < np.mean(greedy_regrets), 'ε-greedy 应远好于纯贪心'
# 衰减通常不差于固定(后期少交探索税)
assert np.mean(eps_decay) <= np.mean(eps_fixed) * 1.2, '衰减 ε 应不明显差于固定'
print('✅ ε-greedy 远胜纯贪心(探索避免卡死)；衰减 ε 后期更省探索税')

## 4 · 固定 ε 的线性铁证：后期每步 regret 恒定

怎么严谨区分「线性」与「次线性」？看**后期每步的 regret 率**。

固定 ε：后期每步 regret **恒定**(Q3率≈Q4率) → 累积线性。下节看 UCB/Thompson 的率会**递减**。

In [ ]:
def quarter_rates(bandit, strategy, T=8000, n_seeds=30, **kw):
    '''返回第3、第4个四分之一的「每步 regret 率」均值。'''
    q = T // 4
    q3s, q4s = [], []
    for sd in range(n_seeds):
        _, reg, _ = run_strategy(bandit, strategy, T=T, seed=sd, **kw)
        q3s.append((reg[3 * q - 1] - reg[2 * q - 1]) / q)   # 第3个1/4每步 regret
        q4s.append((reg[-1] - reg[3 * q - 1]) / q)          # 第4个1/4每步 regret
    return np.mean(q3s), np.mean(q4s)

q3_eps, q4_eps = quarter_rates(bandit, 'epsilon', eps=0.1)
print(f'固定 ε=0.1: Q3 每步 regret={q3_eps:.4f}, Q4 每步 regret={q4_eps:.4f}')
print(f'  比值 Q4/Q3 = {q4_eps / q3_eps:.3f} (≈1 表示恒定 -> 线性累积 regret)')
# 固定 ε 后期每步 regret 应基本恒定(线性)
assert abs(q4_eps / q3_eps - 1.0) < 0.25, '固定 ε 后期每步 regret 应近似恒定(线性标志)'
print('✅ 固定 ε 后期每步 regret 恒定 → 累积 regret 线性(永远交着同样的探索税)')

## 5 · UCB1：置信半径的推导与次线性 regret

UCB1 选 $\arg\max_a[\hat\mu_a + \sqrt{2\ln t / N_a}]$。置信半径来自 **Hoeffding 不等式**：

$P(|\hat\mu_a - \mu_a| \ge u) \le 2e^{-2N_a u^2}$。令失败概率 $= 2t^{-4}$，解出 $u = \sqrt{2\ln t / N_a}$。先验证这个推导。

In [ ]:
import math

def ucb_bonus(t, N):
    return math.sqrt(2 * math.log(t) / N)

# 验证推导：令 Hoeffding 上界 2 exp(-2 N u²) = 2 t^{-4}，解得 u=sqrt(2 ln t/N)
t, N = 1000, 50
u = ucb_bonus(t, N)
hoeffding_tail = 2 * math.exp(-2 * N * u ** 2)
target = 2 * t ** (-4)
print(f'置信半径 u = sqrt(2 ln {t}/{N}) = {u:.4f}')
print(f'此 u 处 Hoeffding 尾概率 = {hoeffding_tail:.2e}, 目标 2/t⁴ = {target:.2e}')
assert abs(hoeffding_tail - target) < 1e-13, 'UCB 半径应由 Hoeffding 反解得到'
# 探索奖励性质：随 N 减小(更确定)，随 t 增大(久未拉则探索)
assert ucb_bonus(1000, 1) > ucb_bonus(1000, 100), '拉得越多探索奖励越小'
assert ucb_bonus(10000, 10) > ucb_bonus(10, 10), '总步数越多探索奖励越大'
print('✅ UCB1 置信半径来自 Hoeffding 不等式；随 N 收缩、随 t 增长')

## 6 · UCB1 的 regret 次线性

UCB1 的 regret 是 $O(\log T)$(Auer 2002)，达 Lai-Robbins 下界量级。验证：后期每步 regret **递减**(Q4 < Q3)。

In [ ]:
ucb_regrets = [run_strategy(bandit, 'ucb', T=5000, seed=sd)[1][-1] for sd in range(20)]
print(f'UCB1 最终 regret = {np.mean(ucb_regrets):.1f} (纯贪心 {np.mean(greedy_regrets):.0f})')
q3_ucb, q4_ucb = quarter_rates(bandit, 'ucb')
print(f'UCB1: Q3 每步 regret={q3_ucb:.4f}, Q4 每步 regret={q4_ucb:.4f} (递减 → 次线性)')
# UCB 远好于贪心
assert np.mean(ucb_regrets) < np.mean(greedy_regrets) / 3, 'UCB 应远好于纯贪心'
# 后期每步 regret 递减(次线性铁证)
assert q4_ucb < q3_ucb, 'UCB 后期每步 regret 应递减(次线性)'
print('✅ UCB1 regret 次线性：后期每步 regret 递减(越来越多拉最优臂)')

## 7 · Thompson sampling：Beta 后验采样

为每臂维护 Beta(α,β) 后验(成功 +α，失败 +β)，每步从各后验采样、选采样最大的臂。

先验证 Beta 后验：随数据增多，后验均值收敛到真值、方差收缩(~1/N)。

In [ ]:
# 验证 Beta-Bernoulli 后验：观测 true p=0.7 的数据，后验均值应收敛
rng = np.random.default_rng(0)
p_true = 0.7; alpha, beta = 1.0, 1.0
for _ in range(2000):
    r = 1.0 if rng.random() < p_true else 0.0
    alpha += r; beta += (1 - r)
post_mean = alpha / (alpha + beta)
post_var = alpha * beta / ((alpha + beta) ** 2 * (alpha + beta + 1))
print(f'Beta 后验均值 = {post_mean:.4f} (真值 {p_true}), 后验 std = {math.sqrt(post_var):.4f}')
assert abs(post_mean - p_true) < 0.03, 'Beta 后验均值应收敛到真值'
assert math.sqrt(post_var) < 0.02, '后验应随数据收紧'

# 跑 Thompson sampling 并验证 regret 次线性
ts_regrets = [run_strategy(bandit, 'thompson', T=5000, seed=sd)[1][-1] for sd in range(20)]
print(f'\nThompson 最终 regret = {np.mean(ts_regrets):.1f}')
q3_ts, q4_ts = quarter_rates(bandit, 'thompson')
print(f'Thompson: Q3 每步 regret={q3_ts:.4f}, Q4 每步 regret={q4_ts:.4f} (递减 → 次线性)')
assert q4_ts < q3_ts, 'Thompson 后期每步 regret 应递减(次线性)'
print('✅ Thompson: Beta 后验采样实现自动校准探索，regret 次线性')

## 8 · 三策略总对比：长跑见真章

把 greedy / 固定ε / UCB / Thompson 放一起比最终 regret。**关键：用长跑 T=20000**——

因为固定 ε 是**线性** regret，短跑(T=5000)时它的累积税还没攒够、看似不差；长跑时它的线性增长必然被 UCB/Thompson 的**次线性**(对数)增长反超。经典结论(Chapelle & Li 2011)：**Thompson 常优于 UCB**。

In [ ]:
T_long = 20000     # 长跑：让线性 regret(固定ε) 充分暴露、被次线性策略反超
print(f"{'策略':<16}{'最终 regret(均值)':>18}{'相对贪心':>12}")
results = {}
for strat, kw in [('greedy', {}), ('epsilon', {'eps': 0.1}),
                  ('ucb', {}), ('thompson', {})]:
    finals = [run_strategy(bandit, strat, T=T_long, seed=sd, **kw)[1][-1] for sd in range(20)]
    results[strat] = np.mean(finals)
    print(f'{strat:<16}{np.mean(finals):>18.1f}{np.mean(finals)/results["greedy"]:>12.2%}')
# 次线性策略(UCB/Thompson)在长跑中远胜线性策略(greedy/固定ε)
assert results['ucb'] < results['epsilon'], '长跑中 UCB(次线性) 应反超固定 ε(线性)'
assert results['thompson'] < results['greedy'] / 10, 'Thompson 应远胜贪心'
# Thompson 常优于 UCB(实证)
assert results['thompson'] <= results['ucb'], 'Thompson 实证常优于 UCB'
print('\n✅ 长跑见真章：次线性的 UCB/Thompson 反超线性的固定ε、碾压贪心；')
print('   Thompson 实证最优(复现 Chapelle & Li 2011)。这就是「regret 增长阶」的威力。')

---
## ✏️ 练习 1：增量均值更新

老虎机的核心是在线估计各臂均值。实现 `update_mean(old_mean, count, new_reward)`：
用增量公式 $\hat\mu \leftarrow \hat\mu + \frac{1}{N}(r - \hat\mu)$ 返回新均值(count 是**包含本次**的拉动次数)。

In [ ]:
def update_mean(old_mean, count, new_reward):
    # TODO: 返回 old_mean + (new_reward - old_mean)/count
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 从 0 开始：第1次拉得 1.0 -> 均值 1.0；第2次拉得 0.0 -> 均值 0.5
m = update_mean(0.0, 1, 1.0)
assert abs(m - 1.0) < 1e-9
m = update_mean(m, 2, 0.0)
assert abs(m - 0.5) < 1e-9
# 增量均值应等于算术平均
rng = np.random.default_rng(0); data = rng.random(100)
m = 0.0
for i, r in enumerate(data, 1):
    m = update_mean(m, i, r)
assert abs(m - data.mean()) < 1e-9, '增量均值应等于 data.mean()'
print('✅ 练习 1 通过：增量均值更新 == 算术平均(无需存全部历史)')

## ✏️ 练习 2：UCB1 选臂

实现 `ucb_select(values, counts, t)`：返回 UCB1 选的臂 $\arg\max_a[\hat\mu_a + \sqrt{2\ln t/N_a}]$。

**注意**：若有臂从未被拉($N_a=0$)，应优先拉它(置信半径无穷大)。

In [ ]:
def ucb_select(values, counts, t):
    # TODO: 若存在 counts==0 的臂，返回其一(如 argmin counts)；
    #       否则返回 argmax(values + sqrt(2 ln t / counts))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 有未拉的臂(臂2 count=0) -> 优先拉它
a = ucb_select(np.array([0.5, 0.9, 0.0]), np.array([10.0, 10.0, 0.0]), t=21)
assert a == 2, '未拉过的臂应被优先探索'
# 全拉过：估计接近时，拉得少的臂(置信半径大)更可能被选
a = ucb_select(np.array([0.5, 0.5]), np.array([100.0, 2.0]), t=103)
assert a == 1, '估计相同时，拉得少(更不确定)的臂应被选(乐观)'
# 估计差距大到盖过置信项时，选高均值臂
a = ucb_select(np.array([0.9, 0.1]), np.array([50.0, 50.0]), t=101)
assert a == 0, '均值明显更高且同样确定时应选它(利用)'
print('✅ 练习 2 通过：UCB1 平衡利用(高均值)与探索(高不确定)')

## ✏️ 练习 3：Thompson 采样选臂

实现 `thompson_select(alpha, beta, rng)`：从每臂的 Beta(α,β) 采一个样本，返回采样值最大的臂。

In [ ]:
def thompson_select(alpha, beta, rng):
    # TODO: samples = rng.beta(alpha, beta)；返回 argmax(samples)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng = np.random.default_rng(0)
# 臂1后验强烈偏高(α大β小)，长期应被选得最多
alpha = np.array([2.0, 50.0, 2.0]); beta = np.array([2.0, 2.0, 2.0])
picks = np.array([thompson_select(alpha, beta, rng) for _ in range(2000)])
freq = np.bincount(picks, minlength=3) / 2000
print('各臂被选频率:', np.round(freq, 3))
assert freq[1] > 0.8, '后验最强的臂应被选得最多(probability matching)'
# 但不确定的臂仍偶有机会(探索)
alpha2 = np.array([1.0, 1.0]); beta2 = np.array([1.0, 1.0])   # 全均匀(全不确定)
picks2 = np.array([thompson_select(alpha2, beta2, rng) for _ in range(2000)])
f2 = np.bincount(picks2, minlength=2) / 2000
assert 0.3 < f2[0] < 0.7, '全不确定时应近似均匀探索两臂'
print('✅ 练习 3 通过：Thompson 按后验概率匹配选臂，不确定时自动多探索')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def update_mean(old_mean, count, new_reward):
    return old_mean + (new_reward - old_mean) / count

In [ ]:
# 练习 2 参考答案
def ucb_select(values, counts, t):
    if (counts == 0).any():
        return int(np.argmin(counts))
    return int(np.argmax(values + np.sqrt(2 * np.log(t) / counts)))

In [ ]:
# 练习 3 参考答案
def thompson_select(alpha, beta, rng):
    return int(np.argmax(rng.beta(alpha, beta)))

---
## 🧪 真实数据胶囊：老虎机做 A/B 测试（网页转化率）

老虎机直接用于工业：**自适应 A/B 测试**。传统 A/B 把流量平均分给各方案、跑完才决策(浪费流量在差方案上)；

用 Thompson sampling 做 A/B，可**边测边把更多流量导向更好的方案**，大幅减少 regret(损失的转化)。
模拟 3 个落地页设计，真实转化率分别 3%/5%/4.5%，看 Thompson 如何自动收敛到最优设计。

In [ ]:
def adaptive_ab_test(conversion_rates, n_visitors=10000, seed=0):
    '''Thompson sampling 做自适应 A/B 测试。返回各方案最终流量分配与累积转化。'''
    rng = np.random.default_rng(seed)
    K = len(conversion_rates)
    alpha = np.ones(K); beta = np.ones(K)
    allocations = np.zeros(K); total_conversions = 0
    for _ in range(n_visitors):
        a = int(np.argmax(rng.beta(alpha, beta)))   # 给这个访客分配方案 a
        converted = 1.0 if rng.random() < conversion_rates[a] else 0.0
        alpha[a] += converted; beta[a] += (1 - converted)
        allocations[a] += 1; total_conversions += converted
    return allocations, total_conversions

rates = np.array([0.03, 0.05, 0.045])    # 方案 1 最优(5%)
alloc, conv = adaptive_ab_test(rates, n_visitors=10000, seed=0)
print('各方案分到的流量:', alloc.astype(int), f'(总 {int(alloc.sum())} 访客)')
print(f'总转化数 = {int(conv)} (转化率 {conv/alloc.sum():.3%})')
# 对比：均匀 A/B 测试(各 1/3 流量)的期望转化
uniform_conv = 10000 / 3 * rates.sum()
best_conv = 10000 * rates.max()
print(f'均匀 A/B 期望转化 = {uniform_conv:.0f}, 全导最优方案 = {best_conv:.0f}')
# Thompson 应把最多流量给最优方案，转化高于均匀分配
assert alloc.argmax() == 1, 'Thompson 应把最多流量导向最优方案(方案1)'
assert conv > uniform_conv, '自适应 A/B 转化应高于均匀 A/B(少浪费在差方案)'
print('✅ Thompson 自适应 A/B：边测边导流，转化高于均匀分配 —— 老虎机的工业价值')

**🧪 胶囊练习**：实现 `regret_vs_oracle(allocations, rates)`：计算这次 A/B 测试相对「事后诸葛(全导最优方案)」的 regret，即 $\sum_a N_a \cdot (\max_k \text{rate}_k - \text{rate}_a)$。这量化了「探索期间损失的转化」。

In [ ]:
def regret_vs_oracle(allocations, rates):
    # TODO: best = rates.max()；返回 sum(allocations[a]*(best-rates[a]))
    raise NotImplementedError

In [ ]:
# 自测
reg = regret_vs_oracle(alloc, rates)
# 均匀分配的 regret 作对比
uniform_alloc = np.full(3, 10000 / 3)
reg_uniform = regret_vs_oracle(uniform_alloc, rates)
print(f'Thompson A/B 的 regret = {reg:.1f} 次转化损失')
print(f'均匀 A/B 的 regret    = {reg_uniform:.1f} 次转化损失')
assert reg >= 0, 'regret 非负'
assert reg < reg_uniform, 'Thompson 的 regret 应小于均匀分配(更少浪费)'
print('✅ 胶囊练习通过：自适应 A/B 的 regret(损失转化)远小于传统均匀 A/B')

In [ ]:
# 📖 胶囊参考答案
def regret_vs_oracle(allocations, rates):
    best = np.max(rates)
    return float(np.sum(allocations * (best - rates)))

### 小结
- **多臂老虎机** = 单状态、K 臂的 RL，把 **explore-exploit 困境**剥离到最纯形式。
- **regret** = T·μ* − 实得 = Σ Δ_a·N_a；好策略 regret **次线性**(O(log T))，盲目策略**线性**。
- **ε-greedy**：盲目均匀探索；固定 ε → 线性 regret(永交探索税)；衰减 ε → 近对数。
- **UCB1**：μ̂ + sqrt(2ln t/N)，「面对不确定的乐观」，Hoeffding 推出置信半径，O(log T) 最优。
- **Thompson**：Beta 后验采样，probability matching，实现简单、实证常优于 UCB。
- **Lai-Robbins 下界**：regret 至少 Ω(log T)，UCB/Thompson 达此最优 —— 探索有代价但可很小。

🎓 **全课完结**：MDP/Bellman → TD/Q-learning → 策略梯度 → Actor-Critic/GAE → 探索。你已把表格 RL 的地基从零夯实——下一站 **C41 深度 RL**(把表格换成神经网络)与 **C22 推理 RL**(RLHF/GRPO)。地基已牢，前沿可期。